# Nonlinear vibration absorber

Run this lab in a kernel with the repository installed. Compare a linear
tuned mass damper with a purely cubic nonlinear energy sink. For the NES,
the attachment force is $k_3(q_1-q_2)^3$ plus relative viscous damping.
Its response depends on drive frequency and amplitude.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from modal_lab import (
    absorber_system, damped_oscillator, harmonic_force, linear_frf,
    nonlinear_frf, power_spectrum, simulate, suppression_bands,
)
from modal_lab.plotting import plot_frf

## 1. Compare detuned fundamental responses

Drive the primary with the same force in all three cases. Linear curves
come from the dynamic stiffness matrix. Each nonlinear sweep point
starts from rest, discards initial cycles, and projects the final
displacement onto the drive sinusoid. Longer transients or another
attractor can change the curve. Try changing the force or cubic stiffness.

In [ ]:
omega = np.linspace(0.65, 1.35, 21)
force = 0.1
primary = damped_oscillator(zeta=0.03)
nes = absorber_system("nes", cubic=3)
bare = linear_frf(primary, omega, load=force)[:, 0]
tmd = linear_frf(absorber_system(), omega, load=[force, 0])[:, 0]
cubic = nonlinear_frf(nes, omega, load=[force, 0])[:, 0]
fig = plot_frf({"Uncontrolled": (omega, bare), "Linear TMD": (omega, tmd),
               "Cubic NES": (omega, cubic)})
for name, response in (("TMD", tmd), ("NES", cubic)):
    bands = suppression_bands(omega, bare, response, db=3)
    width = sum(hi - lo for lo, hi in bands)
    print(f"{name}: 3 dB suppression width = {width:.3f} rad/s")
plt.show()

## 2. Look for harmonics

Record the attachment relative displacement at one frequency. Analyze
the final 12 complete cycles on a uniform grid. Peaks away from the
drive frequency can indicate nonlinear distortion or more complicated
motion. Compare with the linear TMD when driven at the same frequency.

In [ ]:
drive = 1.0
period = 2 * np.pi / drive
samples_per_cycle = 96
steady = (50 + np.arange(12 * samples_per_cycle) / samples_per_cycle) * period
times = np.r_[0.0, steady]
fig, ax = plt.subplots(figsize=(7.5, 3.6), layout="constrained")
for name, model in (("Linear TMD", absorber_system()), ("Cubic NES", nes)):
    run = simulate(model, times, force=harmonic_force([force, 0], drive), max_step=period / 16)
    relative = run.displacement[1:, 0] - run.displacement[1:, 1]
    frequency, power = power_spectrum(steady, relative, window="boxcar")
    ax.semilogy(2 * np.pi * frequency / drive, np.maximum(power, 1e-16), label=name)
ax.set(xlim=(0, 7), ylim=(1e-11, 10), xlabel="Frequency / drive frequency",
       ylabel="Relative displacement power (m²)")
ax.legend()
plt.show()